# Evaluation — Head-to-Head and Round-Robin

Interactive tool for comparing Connect-4 agents. Pick two or more models, set the number of
games, and hit Run. The library code lives in `src/eval.py`; this notebook is just the
control panel.

**Selection rules:**
- **2 models** -> single head-to-head match (alternating first-player)
- **3 or more models** -> round-robin (every pair plays every other pair)
- **0 or 1 selected** -> error

All `ModelAgent`s below use **greedy** move selection (argmax) with **tactical overrides
on** (immediate-win and immediate-block) — mirroring how the agent will be deployed for
the tournament. Instantiate a copy with `use_tactics=False` if you want the pure
model-vs-model comparison for analysis.

In [ ]:
# ── Cell 1: Imports and model loading ────────────────────────────────────────
import os, sys

# Make the repo root importable from this notebook (inside notebooks/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src import model_loader
from src.eval import (
    ModelAgent, RandomAgent,
    play_match, run_round_robin,
    format_result, round_robin_to_dataframe,
)

models = model_loader.load_all_models()
print(f"\nLoaded {len(models)} models.")

In [ ]:
# ── Cell 2: Wrap each model as an Agent ──────────────────────────────────────
# Defaults: greedy=True (argmax), use_tactics=True (immediate win / immediate block).
# These are the settings the deployed tournament agent will use.

agents = {
    name: ModelAgent(wrapper, greedy=True, use_tactics=True)
    for name, wrapper in models.items()
}
agents["random"] = RandomAgent()

print(f"Built {len(agents)} agents:")
for name in agents:
    print(f"  - {name}")

In [ ]:
# ── Cell 3: Interactive evaluation UI ────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

# One checkbox per agent
checkboxes = {
    name: widgets.Checkbox(value=False, description=name, indent=False)
    for name in agents
}

n_games_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='N games per pair:',
    style={'description_width': 'initial'},
    continuous_update=False,
)

run_button = widgets.Button(
    description='Run evaluation',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px'),
)

output = widgets.Output()

def on_run_clicked(b):
    with output:
        clear_output()
        selected = [name for name, cb in checkboxes.items() if cb.value]
        n = len(selected)

        if n < 2:
            print(f"ERROR: select at least 2 models ({n} currently selected).")
            return

        n_games = n_games_slider.value

        if n == 2:
            a, b_ = selected
            print(f"Head-to-head: {a} vs {b_} ({n_games} games, alternating first)\n")
            result = play_match(agents[a], agents[b_], n_games=n_games)
            print(format_result(result))
        else:
            print(f"Round-robin: {n} agents, {n_games} games per pair\n")
            subset = {k: agents[k] for k in selected}
            results = run_round_robin(subset, n_games=n_games)
            df = round_robin_to_dataframe(results, selected)

            print("Win-rate matrix — row agent's win rate when playing column agent:")
            print((df * 100).round(1).fillna('--').to_string())
            print()

            # Overall ranking by mean win rate across the other agents
            ranking = df.mean(axis=1).sort_values(ascending=False)
            print("Overall ranking (mean win rate across all opponents):")
            for i, (name, wr) in enumerate(ranking.items(), 1):
                print(f"  {i}. {name:30s}  {wr:.1%}")

run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Select 2 or more models</h3>"),
    widgets.VBox(list(checkboxes.values())),
    widgets.HTML("<br>"),
    n_games_slider,
    run_button,
    widgets.HTML("<hr>"),
    output,
])
display(ui)

In [ ]:
# ── Cell 4 (optional): Programmatic use, no UI ───────────────────────────────
# Useful for scripting a specific matchup or saving results to disk.

# Example: quick M1 vs every baseline
# results = {}
# opponents = ["random", "stiles_cnn", "luke_cnn", "zan_transformer"]
# for opp in opponents:
#     r = play_match(agents["m1"], agents[opp], n_games=100)
#     results[opp] = r
#     print(format_result(r))
#     print()